# 02 — PD Dataset Construction

This notebook builds the Probability of Default (PD) modelling dataset from the validated Master Credit Risk Dataset.

Target convention used in the course:

- `good_bad = 0`: bad borrower;
- `good_bad = 1`: good borrower.

Bad statuses:

- `Charged Off`;
- `Default`;
- `Does not meet the credit policy. Status:Charged Off`;
- `Late (31-120 days)`.

Every other observed status is classified as good.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


## 1. Resolve project paths


In [2]:
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports" / "pd_dataset"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Processed data directory: {PROCESSED_DIR.resolve()}")
print(f"PD report directory: {REPORT_DIR.resolve()}")


Project root: C:\Users\Platini AGOUANET\Mes Dossiers lourds\risk-credit-scoring-new
Processed data directory: C:\Users\Platini AGOUANET\Mes Dossiers lourds\risk-credit-scoring-new\data\processed
PD report directory: C:\Users\Platini AGOUANET\Mes Dossiers lourds\risk-credit-scoring-new\reports\pd_dataset


## 2. Load the Master Dataset

Parquet is preferred because it preserves data types. CSV is used as a fallback.


In [3]:
MASTER_PARQUET_PATH = PROCESSED_DIR / "master_credit_risk_dataset.parquet"
MASTER_CSV_PATH = PROCESSED_DIR / "master_credit_risk_dataset.csv"

if MASTER_PARQUET_PATH.exists():
    loan_data = pd.read_parquet(MASTER_PARQUET_PATH)
    source_path = MASTER_PARQUET_PATH
elif MASTER_CSV_PATH.exists():
    loan_data = pd.read_csv(MASTER_CSV_PATH, low_memory=False)
    source_path = MASTER_CSV_PATH
else:
    raise FileNotFoundError(
        "Master Dataset not found. Run 01_Master_Dataset_Construction_Final.ipynb first."
    )

print(f"Loaded: {source_path.resolve()}")
print(f"Shape: {loan_data.shape}")
display(loan_data.head())


Loaded: C:\Users\Platini AGOUANET\Mes Dossiers lourds\risk-credit-scoring-new\data\processed\master_credit_risk_dataset.parquet
Shape: (466285, 69)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,emp_length_missing,emp_length_years,credit_history_months,funding_ratio,investor_funding_ratio,investor_loan_ratio,loan_to_income_ratio,installment_to_income_ratio,open_account_ratio,credit_inquiry_rate,delinquency_rate,emp_title_missing,loan_burden_interest,mths_since_last_record_is_missing,mths_since_last_major_derog_is_missing,mths_since_last_delinq_is_missing,open_account_inconsistency
0,1077501,1296599,5000,5000,4975.0,36,10.65,162.87,B,B2,None,10+ years,RENT,24000.0,Verified,2011-12-01,Fully Paid,credit_card,Computer,860xx,AZ,27.65,0.0,1985-01-01,1.0,-1.0,-1.0,3.0,0.0,13648,83.7,9.0,f,0.0,0.0,5861.071414,5831.78,5000.00,861.07,0.00,0.00,0.00,2015-01-01,171.62,NaT,2016-01-01,0.0,-1.0,0.0,NaN,NaN,NaN,0,10,323,1.0,0.995,0.995,0.208333,0.081435,0.333333,0.037152,0.0,1,2.218750,1,1,1,0
1,1077430,1314167,2500,2500,2500.0,60,15.27,59.83,C,C4,Ryder,< 1 year,RENT,30000.0,Source Verified,2011-12-01,Charged Off,car,bike,309xx,GA,1.00,0.0,1999-04-01,5.0,-1.0,-1.0,3.0,0.0,1687,9.4,4.0,f,0.0,0.0,1008.710000,1008.71,456.46,435.17,0.00,117.08,1.11,2013-04-01,119.66,NaT,2013-09-01,0.0,-1.0,0.0,NaN,NaN,NaN,0,0,152,1.0,1.000,1.000,0.083333,0.023932,0.750000,0.394737,0.0,0,1.272500,1,1,1,0
2,1077175,1313524,2400,2400,2400.0,36,15.96,84.33,C,C5,None,10+ years,RENT,12252.0,Not Verified,2011-12-01,Fully Paid,small_business,real estate business,606xx,IL,8.72,0.0,2001-11-01,2.0,-1.0,-1.0,2.0,0.0,2956,98.5,10.0,f,0.0,0.0,3003.653644,3003.65,2400.00,603.65,0.00,0.00,0.00,2014-06-01,649.91,NaT,2016-01-01,0.0,-1.0,0.0,NaN,NaN,NaN,0,10,121,1.0,1.000,1.000,0.195886,0.082595,0.200000,0.198347,0.0,1,3.126347,1,1,1,0
3,1076863,1277178,10000,10000,10000.0,36,13.49,339.31,C,C1,AIR RESOURCES BOARD,10+ years,RENT,49200.0,Source Verified,2011-12-01,Fully Paid,other,personel,917xx,CA,20.00,0.0,1996-02-01,1.0,35.0,-1.0,10.0,0.0,5598,21.0,37.0,f,0.0,0.0,12226.302210,12226.30,10000.00,2209.33,16.97,0.00,0.00,2015-01-01,357.48,NaT,2015-01-01,0.0,-1.0,0.0,NaN,NaN,NaN,0,10,190,1.0,1.000,1.000,0.203252,0.082759,0.270270,0.063158,0.0,0,2.741870,1,1,0,0
4,1075358,1311748,3000,3000,3000.0,60,12.69,67.79,B,B5,University Medical Group,1 year,RENT,80000.0,Source Verified,2011-12-01,Current,other,Personal,972xx,OR,17.94,0.0,1996-01-01,0.0,38.0,-1.0,15.0,0.0,27783,53.9,38.0,f,766.9,766.9,3242.170000,3242.17,2233.10,1009.07,0.00,0.00,0.00,2016-01-01,67.79,2016-02-01,2016-01-01,0.0,-1.0,0.0,NaN,NaN,NaN,0,1,191,1.0,1.000,1.000,0.037500,0.010169,0.394737,0.0,0.0,0,0.475875,1,1,0,0


## 3. Inspect `loan_status`


In [4]:
if "loan_status" not in loan_data.columns:
    raise KeyError("The Master Dataset does not contain 'loan_status'.")

loan_status_distribution = (
    loan_data["loan_status"]
    .value_counts(dropna=False)
    .rename_axis("loan_status")
    .to_frame("record_count")
)
loan_status_distribution["percentage"] = (
    loan_status_distribution["record_count"] / len(loan_data) * 100
)

display(loan_status_distribution)


,record_count,percentage
loan_status,,
Current,224226,48.087757
Fully Paid,184739,39.619332
Charged Off,42475,9.109236
Late (31-120 days),6900,1.479782
In Grace Period,3146,0.674695
Does not meet the credit policy. Status:Fully Paid,1988,0.426349
Late (16-30 days),1218,0.261214
Default,832,0.178432
Does not meet the credit policy. Status:Charged Off,761,0.163205


## 4. Create the binary target

All statuses not listed as bad are assigned to the good class.


In [5]:
BAD_STATUSES = {
    "Charged Off",
    "Default",
    "Does not meet the credit policy. Status:Charged Off",
    "Late (31-120 days)",
}

loan_data["good_bad"] = np.where(
    loan_data["loan_status"].isin(BAD_STATUSES),
    0,
    1,
).astype("int8")

print("Target created: 0 = bad, 1 = good")
display(pd.crosstab(loan_data["loan_status"], loan_data["good_bad"], margins=True))


Target created: 0 = bad, 1 = good


good_bad,0,1,All
loan_status,,,
Charged Off,42475,0,42475
Current,0,224226,224226
Default,832,0,832
Does not meet the credit policy. Status:Charged Off,761,0,761
Does not meet the credit policy. Status:Fully Paid,0,1988,1988
Fully Paid,0,184739,184739
In Grace Period,0,3146,3146
Late (16-30 days),0,1218,1218
Late (31-120 days),6900,0,6900


## 5. Validate target distribution


In [6]:
target_distribution = (
    loan_data["good_bad"]
    .value_counts()
    .sort_index()
    .rename_axis("good_bad")
    .to_frame("record_count")
)
target_distribution["class_label"] = target_distribution.index.map({0: "Bad", 1: "Good"})
target_distribution["percentage"] = (
    target_distribution["record_count"] / len(loan_data) * 100
)

display(target_distribution)
#print(f"Bad rate:  {(loan_data['good_bad'] == 0).mean():.2%}")
#print(f"Good rate: {(loan_data['good_bad'] == 1).mean():.2%}")


,record_count,class_label,percentage
good_bad,,,
0,50968,Bad,10.930654
1,415317,Good,89.069346


## 6. Define variables excluded from PD modelling

The Master Dataset remains unchanged. Only the PD modelling dataset excludes post-origination variables, direct outcome proxies, identifiers and unsuitable free-text fields.


In [7]:
LEAKAGE_CANDIDATES = [
    "loan_status",
    "out_prncp",
    "out_prncp_inv",
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_d",
    "last_pymnt_amnt",
    "next_pymnt_d",
    "last_credit_pull_d",
    "sub_grade",
    "emp_length",
]

NON_MODELLING_COLUMNS = [
    "id",
    "member_id",
    "url",
    "desc",
    "title",
    "emp_title",
    "zip_code",
]

available_leakage_columns = [c for c in LEAKAGE_CANDIDATES if c in loan_data.columns]
available_non_modelling_columns = [c for c in NON_MODELLING_COLUMNS if c in loan_data.columns]

#print("Leakage/post-origination columns to remove:")
#for column in available_leakage_columns:
#    print(f" - {column}")

#print("\nIdentifiers/free-text columns to remove:")
#for column in available_non_modelling_columns:
#    print(f" - {column}")


## 7. Construct the PD modelling dataset


In [8]:
columns_to_remove = sorted(
    set(available_leakage_columns + available_non_modelling_columns) - {"good_bad"}
)

pd_data = loan_data.drop(columns=columns_to_remove).copy()

assert "good_bad" in pd_data.columns
assert "loan_status" not in pd_data.columns

print(f"Master Dataset shape: {loan_data.shape}")
print(f"PD Dataset shape:     {pd_data.shape}")
print(f"Columns removed:      {len(columns_to_remove)}")

exclusion_report = pd.DataFrame({"column": columns_to_remove})
exclusion_report["reason"] = exclusion_report["column"].map(
    lambda column: (
        "Potential post-origination or target leakage"
        if column in available_leakage_columns
        else "Identifier, free text, or unsuitable high-cardinality field"
    )
)

display(pd_data.head())
display(exclusion_report)

Master Dataset shape: (466285, 70)
PD Dataset shape:     (466285, 49)
Columns removed:      21


,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,home_ownership,annual_inc,verification_status,issue_d,purpose,addr_state,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,emp_length_missing,emp_length_years,credit_history_months,funding_ratio,investor_funding_ratio,investor_loan_ratio,loan_to_income_ratio,installment_to_income_ratio,open_account_ratio,credit_inquiry_rate,delinquency_rate,emp_title_missing,loan_burden_interest,mths_since_last_record_is_missing,mths_since_last_major_derog_is_missing,mths_since_last_delinq_is_missing,open_account_inconsistency,good_bad
0,5000,5000,4975.0,36,10.65,162.87,B,RENT,24000.0,Verified,2011-12-01,credit_card,AZ,27.65,0.0,1985-01-01,1.0,-1.0,-1.0,3.0,0.0,13648,83.7,9.0,f,0.0,-1.0,0.0,NaN,NaN,NaN,0,10,323,1.0,0.995,0.995,0.208333,0.081435,0.333333,0.037152,0.0,1,2.218750,1,1,1,0,1
1,2500,2500,2500.0,60,15.27,59.83,C,RENT,30000.0,Source Verified,2011-12-01,car,GA,1.00,0.0,1999-04-01,5.0,-1.0,-1.0,3.0,0.0,1687,9.4,4.0,f,0.0,-1.0,0.0,NaN,NaN,NaN,0,0,152,1.0,1.000,1.000,0.083333,0.023932,0.750000,0.394737,0.0,0,1.272500,1,1,1,0,0
2,2400,2400,2400.0,36,15.96,84.33,C,RENT,12252.0,Not Verified,2011-12-01,small_business,IL,8.72,0.0,2001-11-01,2.0,-1.0,-1.0,2.0,0.0,2956,98.5,10.0,f,0.0,-1.0,0.0,NaN,NaN,NaN,0,10,121,1.0,1.000,1.000,0.195886,0.082595,0.200000,0.198347,0.0,1,3.126347,1,1,1,0,1
3,10000,10000,10000.0,36,13.49,339.31,C,RENT,49200.0,Source Verified,2011-12-01,other,CA,20.00,0.0,1996-02-01,1.0,35.0,-1.0,10.0,0.0,5598,21.0,37.0,f,0.0,-1.0,0.0,NaN,NaN,NaN,0,10,190,1.0,1.000,1.000,0.203252,0.082759,0.270270,0.063158,0.0,0,2.741870,1,1,0,0,1
4,3000,3000,3000.0,60,12.69,67.79,B,RENT,80000.0,Source Verified,2011-12-01,other,OR,17.94,0.0,1996-01-01,0.0,38.0,-1.0,15.0,0.0,27783,53.9,38.0,f,0.0,-1.0,0.0,NaN,NaN,NaN,0,1,191,1.0,1.000,1.000,0.037500,0.010169,0.394737,0.0,0.0,0,0.475875,1,1,0,0,1


,column,reason
0,collection_recovery_fee,Potential post-origination or target leakage
1,emp_length,Potential post-origination or target leakage
2,emp_title,"Identifier, free text, or unsuitable high-card..."
3,id,"Identifier, free text, or unsuitable high-card..."
4,last_credit_pull_d,Potential post-origination or target leakage
5,last_pymnt_amnt,Potential post-origination or target leakage
6,last_pymnt_d,Potential post-origination or target leakage
7,loan_status,Potential post-origination or target leakage
8,member_id,"Identifier, free text, or unsuitable high-card..."
9,next_pymnt_d,Potential post-origination or target leakage


## 8. Final validation


In [9]:
duplicate_columns = pd_data.columns[pd_data.columns.duplicated()].tolist()
target_missing = int(pd_data["good_bad"].isna().sum())
unexpected_target_values = sorted(set(pd_data["good_bad"].unique()) - {0, 1})

validation_summary = pd.DataFrame({
    "metric": [
        "rows",
        "columns",
        "duplicate_rows",
        "duplicate_column_names",
        "missing_target_values",
        "good_records",
        "bad_records",
    ],
    "value": [
        pd_data.shape[0],
        pd_data.shape[1],
        int(pd_data.duplicated().sum()),
        len(duplicate_columns),
        target_missing,
        int((pd_data["good_bad"] == 1).sum()),
        int((pd_data["good_bad"] == 0).sum()),
    ],
})

display(validation_summary)

assert not duplicate_columns, "Duplicate column names detected."
assert target_missing == 0, "The target contains missing values."
assert not unexpected_target_values, f"Unexpected target values: {unexpected_target_values}"
assert pd_data["good_bad"].nunique() == 2, "The target must contain both classes."

print("PD Dataset validation passed.")


,metric,value
0,rows,466285
1,columns,49
2,duplicate_rows,0
3,duplicate_column_names,0
4,missing_target_values,0
5,good_records,415317
6,bad_records,50968


PD Dataset validation passed.


## 9. Produce quality and exclusion reports


In [10]:
pd_quality_report = pd.DataFrame({
    "dtype": pd_data.dtypes.astype(str),
    "missing_values": pd_data.isna().sum(),
    "missing_percentage": pd_data.isna().mean().mul(100),
    "unique_values": pd_data.nunique(dropna=False),
}).sort_values(
    by=["missing_percentage", "unique_values"],
    ascending=[False, False],
)

display(pd_quality_report.head(30))


,dtype,missing_values,missing_percentage,unique_values
tot_cur_bal,float64,70276,15.071469,220691
total_rev_hi_lim,float64,70276,15.071469,14613
tot_coll_amt,float64,70276,15.071469,6322
emp_length_years,Int8,21008,4.505399,12
revol_util,float64,340,0.072917,1270
collections_12_mths_ex_med,float64,145,0.031097,10
credit_inquiry_rate,Float64,29,0.006219,2399
open_account_ratio,float64,29,0.006219,1387
credit_history_months,Int16,29,0.006219,691
earliest_cr_line,datetime64[ns],29,0.006219,665


## 10. Export the PD dataset

The exported dataset is not yet split and has not undergone binning, WoE transformation, IV selection, statistical imputation or standardisation.


In [11]:
PD_PARQUET_PATH = PROCESSED_DIR / "pd_modeling_dataset.parquet"
PD_CSV_PATH = PROCESSED_DIR / "pd_modeling_dataset.csv"
TARGET_REPORT_PATH = REPORT_DIR / "target_distribution.csv"
STATUS_REPORT_PATH = REPORT_DIR / "loan_status_mapping.csv"
QUALITY_REPORT_PATH = REPORT_DIR / "pd_dataset_quality_report.csv"
EXCLUSION_REPORT_PATH = REPORT_DIR / "pd_excluded_columns.csv"
VALIDATION_REPORT_PATH = REPORT_DIR / "pd_dataset_validation_summary.csv"

try:
    pd_data.to_parquet(PD_PARQUET_PATH, index=False)
    parquet_exported = True
except ImportError as error:
    parquet_exported = False
    print("Parquet export skipped. Install pyarrow or fastparquet to enable it.")
    print(f"Details: {error}")

pd_data.to_csv(PD_CSV_PATH, index=False)
target_distribution.to_csv(TARGET_REPORT_PATH, index=True)

status_mapping_report = loan_status_distribution.copy()
status_mapping_report["good_bad"] = status_mapping_report.index.map(
    lambda status: 0 if status in BAD_STATUSES else 1
)
status_mapping_report["class_label"] = status_mapping_report["good_bad"].map({0: "Bad", 1: "Good"})
status_mapping_report.to_csv(STATUS_REPORT_PATH, index=True)

pd_quality_report.to_csv(QUALITY_REPORT_PATH, index=True, index_label="column")
exclusion_report.to_csv(EXCLUSION_REPORT_PATH, index=False)
validation_summary.to_csv(VALIDATION_REPORT_PATH, index=False)

print("=" * 70)
print("PD DATASET EXPORT COMPLETED")
print("=" * 70)
if parquet_exported:
    print(f"PD Dataset (Parquet): {PD_PARQUET_PATH.resolve()}")
print(f"PD Dataset (CSV):     {PD_CSV_PATH.resolve()}")
print(f"Target report:        {TARGET_REPORT_PATH.resolve()}")
print(f"Status mapping:       {STATUS_REPORT_PATH.resolve()}")
print(f"Quality report:       {QUALITY_REPORT_PATH.resolve()}")
print(f"Exclusion report:     {EXCLUSION_REPORT_PATH.resolve()}")
print(f"Validation report:    {VALIDATION_REPORT_PATH.resolve()}")


PD DATASET EXPORT COMPLETED
PD Dataset (Parquet): C:\Users\Platini AGOUANET\Mes Dossiers lourds\risk-credit-scoring-new\data\processed\pd_modeling_dataset.parquet
PD Dataset (CSV):     C:\Users\Platini AGOUANET\Mes Dossiers lourds\risk-credit-scoring-new\data\processed\pd_modeling_dataset.csv
Target report:        C:\Users\Platini AGOUANET\Mes Dossiers lourds\risk-credit-scoring-new\reports\pd_dataset\target_distribution.csv
Status mapping:       C:\Users\Platini AGOUANET\Mes Dossiers lourds\risk-credit-scoring-new\reports\pd_dataset\loan_status_mapping.csv
Quality report:       C:\Users\Platini AGOUANET\Mes Dossiers lourds\risk-credit-scoring-new\reports\pd_dataset\pd_dataset_quality_report.csv
Exclusion report:     C:\Users\Platini AGOUANET\Mes Dossiers lourds\risk-credit-scoring-new\reports\pd_dataset\pd_excluded_columns.csv
Validation report:    C:\Users\Platini AGOUANET\Mes Dossiers lourds\risk-credit-scoring-new\reports\pd_dataset\pd_dataset_validation_summary.csv


## Next step

The next notebook will:

1. separate predictors and `good_bad`;
2. create a stratified train/test split;
3. save the split datasets;
4. learn binning rules, WoE mappings and Information Value on the training set only;
5. apply the learned rules unchanged to the test set.

Because `good_bad = 1` represents a good borrower:

- `predict_proba(...)[..., 1]` estimates the probability of being good;
- probability of default can later be calculated as `1 - probability_good`.
